<a href="https://colab.research.google.com/github/krishnanithesh22-stack/Calculating-Distance-between-Earth-and-Other-Planets/blob/main/Calculating_Distance_between_Earth_and_Other_Planets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install skyfield

In [8]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from skyfield.api import load
from datetime import datetime, timezone

In [9]:
eph = load('de421.bsp')
ts = load.timescale()
t = ts.now()

sun = eph['sun']
earth= eph['earth']
moon = eph['moon']

planets = {
    'Mercury': eph['mercury'],
    'Venus': eph['venus'],
    'Mars': eph['mars'],
    'Jupiter': eph['jupiter barycenter'],
    'Saturn': eph['saturn barycenter'],
    'Uranus': eph['uranus barycenter'],
    'Neptune': eph['neptune barycenter']
}

print("Astronomical data loaded successfully!")
print("Date and time (UTC):", t.utc_iso())
print("Number of planets:", len(planets))

Astronomical data loaded successfully!
Date and time (UTC): 2026-11-22T12:46:06Z
Number of planets: 7


In [10]:
moon_distance = earth.at(t).observe(moon).distance()
moon_km = moon_distance.km
moon_million_km = moon_km / 1_000_000
moon_au = moon_distance.au

print("-" * 70)
print("EARTH TO MOON")
print(f"Distance: {moon_km:,.2f} km")
print(f"Million km: {moon_million_km:.4f}")
print(f"Astronomical Units: {moon_au:.8f} AU")
print("-" * 70)

----------------------------------------------------------------------
EARTH TO MOON
Distance: 368,864.09 km
Million km: 0.3689
Astronomical Units: 0.00246570 AU
----------------------------------------------------------------------


In [15]:
results = []
sun_distance = earth.at(t).observe(sun).distance()

results.append({
    'Celestial Body': 'Sun',
    'Distance (km)': sun_distance.km,
    'Distance (Million km)': sun_distance.km / 1_000_000,
    'Distance (AU)': sun_distance.au
})

moon_distance = earth.at(t).observe(moon).distance()
moon_km = moon_distance.km
moon_million_km = moon_km / 1_000_000
moon_au = moon_distance.au

results.append({
    'Celestial Body': 'Moon',
    'Distance (km)': moon_km,
    'Distance (Million km)': moon_million_km,
    'Distance (AU)': moon_au
})

for name, body in planets.items():

    distance = earth.at(t).observe(body).distance()

    results.append({
        'Celestial Body': name,
        'Distance (km)': distance.km,
        'Distance (Million km)': distance.km / 1_000_000,
        'Distance (AU)': distance.au
    })


df = pd.DataFrame(results)

print("-" * 75)
print("       EARTH TO CELESTIAL BODY DISTANCE CALCULATOR")
print("-" * 75)

print("Observation Date (UTC):", t.utc_iso())
print("Total Celestial Bodies:", len(df))
print("-" * 75)

display(df.round({
    'Distance (km)': 2,
    'Distance (Million km)': 4,
    'Distance (AU)': 6
}))

print("\nDistance calculations completed successfully!")



---------------------------------------------------------------------------
       EARTH TO CELESTIAL BODY DISTANCE CALCULATOR
---------------------------------------------------------------------------
Observation Date (UTC): 2026-11-22T12:46:06Z
Total Celestial Bodies: 9
---------------------------------------------------------------------------


,Celestial Body,Distance (km),Distance (Million km),Distance (AU)
0,Sun,1.477564e+08,147.7564,0.987690
1,Moon,3.688641e+05,0.3689,0.002466
2,Mercury,1.554239e+08,155.4239,1.038945
3,Venus,5.568866e+07,55.6887,0.372256
4,Mars,1.871708e+08,187.1708,1.251160
5,Jupiter,7.726428e+08,772.6428,5.164798
6,Saturn,1.313319e+09,1313.3187,8.778994
7,Uranus,2.759468e+09,2759.4677,18.445902
8,Neptune,4.390562e+09,4390.5618,29.349093



Distance calculations completed successfully!


In [17]:
chart_df = df[df['Celestial Body'] != 'Moon'].copy()

fig = go.Figure()

fig.add_trace(go.Bar(
    x = chart_df['Celestial Body'],
    y = chart_df['Distance (Million km)'],
    text = chart_df['Distance (Million km)'].round(2),
    textposition = 'auto',
    name = 'Distance'
))

fig.update_layout(
    title = 'Earth to Sun and Planetary Distances',
    xaxis_title = 'Celestial Bodies',
    yaxis_title = 'Distance (Million Kilometres)',
    template = 'plotly_dark',
    height = 700
)

In [18]:

# Calculate heliocentric positions in AU
bodies_3d = {
    'Mercury': eph['mercury'],
    'Venus': eph['venus'],
    'Earth': earth,
    'Moon': moon,
    'Mars': eph['mars'],
    'Jupiter': eph['jupiter barycenter'],
    'Saturn': eph['saturn barycenter'],
    'Uranus': eph['uranus barycenter'],
    'Neptune': eph['neptune barycenter']
}

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode='markers+text',
    marker=dict(size=12, color='yellow'),
    text=['SUN'],
    textposition='top center',
    name='Sun'
))

for name, body in bodies_3d.items():

    position = sun.at(t).observe(body).position.au

    fig.add_trace(go.Scatter3d(
        x=[position[0]],
        y=[position[1]],
        z=[position[2]],
        mode='markers+text',
        marker=dict(size=7 if name != 'Moon' else 4),
        text=[name],
        textposition='top center',
        name=name
    ))

fig.update_layout(
    title = '3D Interactive Solar System - Real Astronomical Positions',
    template = 'plotly_dark',
    height = 900,
    scene=dict(
        xaxis_title='X Position (AU)',
        yaxis_title='Y Position (AU)',
        zaxis_title='Z Position (AU)',
        aspectmode='data',
        bgcolor='black'
    )
)

fig.show()


In [19]:
moon_position = earth.at(t).observe(moon).position.km

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode='markers+text',
    marker=dict(size=12, color='blue'),
    text=['EARTH'],
    textposition='top center',
    name='Earth'
))

fig.add_trace(go.Scatter3d(
    x=[moon_position[0]],
    y=[moon_position[1]],
    z=[moon_position[2]],
    mode='markers+text',
    marker=dict(size=8, color='lightgray'),
    text=['MOON'],
    textposition='top center',
    name='Moon'
))

fig.update_layout(
    title='Earth-Moon Distance: 3D Close-up',
    template='plotly_dark',
    height=650,
    scene=dict(
        xaxis_title='X (km)',
        yaxis_title='Y (km)',
        zaxis_title='Z (km)',
        aspectmode='data',
        bgcolor='black'
    )
)

fig.show()
